In [1]:
%pip install plotly
%pip install streamlit 
%pip install requests
%pip install pandas

import requests
import streamlit as st 
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go

url = "https://live.euroleague.net/api/v2/private"

credentials = {"user": "" , "password" : ""}

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


/Users/alexis/Documents/ENSAE/AlgoEtProgrammation/ProjetsPerso/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def get_camecodes(season):
    try :
        r = requests.get(f"{url}/games", params = {**credentials, "seasoncode" : f"E{season}"})
        r.raise_for_status() 

        games_data = r.json()
        dico_gamecodes = {}

        for game in games_data["games"]:
            code = game.get("gamecode")
            teamA = game.get("team1")
            teamB = game.get("team2")
            if code :
                dico_gamecodes[code] = {"teamA": teamA, "teamB": teamB}

        return dico_gamecodes 
      
    except Exception as e:
        print(f"Erreur lors de la récupération des gamecodes : {e}")
        return []

On récupère les gamecodes mais on prend aussi les équipes qui jouent.

In [3]:
def get_fg(season, gamecode):
    try :
        r = requests.get(f"{url}/shootingchart", params = {**credentials, "gamecode" : gamecode, "seasoncode" : f"E{season}"})
        r.raise_for_status()
        data = r.json()
        
        teamA = data["mainData"]["CodeTeamA"]
        teamB = data["mainData"]["CodeTeamB"]

        dA = {"C" : (0,0), "D" : (0,0), "E" : (0,0), "F" : (0,0), "G" : (0,0), "H" : (0,0), "I" : (0,0), "J" : (0,0), "K" : (0,0), "L" : (0,0), }
        dB = {"C" : (0,0), "D" : (0,0), "E" : (0,0), "F" : (0,0), "G" : (0,0), "H" : (0,0), "I" : (0,0), "J" : (0,0), "K" : (0,0), "L" : (0,0), }
        donnees = data["points"]["Rows"]

        for play in donnees : 

            if play["CodeTeam"] == teamA :
                if play["ID_ACTION"] == "2FGA" or play["ID_ACTION"] == "3FGA" :
                    dA[play["ZONE"]] = (dA[play["ZONE"]][0], dA[play["ZONE"]][1]+1)
                elif play["ID_ACTION"] == "2FGM" or play["ID_ACTION"] == "3FGM" :
                    dA[play["ZONE"]] = (dA[play["ZONE"]][0]+1, dA[play["ZONE"]][1]+1)

            elif play["CodeTeam"] == teamB :
                if play["ID_ACTION"] == "2FGA" or play["ID_ACTION"] == "3FGA" :
                    dB[play["ZONE"]] = (dB[play["ZONE"]][0], dB[play["ZONE"]][1]+1)
                elif play["ID_ACTION"] == "2FGM" or play["ID_ACTION"] == "3FGM" :
                    dB[play["ZONE"]] = (dB[play["ZONE"]][0]+1, dB[play["ZONE"]][1]+1)
        
        return dA, dB, teamA, teamB
    
    except Exception as e:
        print(f"Erreur lors de la récupération des données de tir : {e}")
        return {}, {}, "", ""

On récupère le nombre de tirs réussis et ratés par zone dans un dictionnaire mais ici on fait match par match car on parcours le match action par action.

In [4]:
def get_fg_all_season(season) : 
    gamecodes = get_camecodes(season)
    dico = {}

    for gamecode in gamecodes.keys() : 
        dA, dB, code_teamA, code_teamB = get_fg(season, gamecode)

        if code_teamA not in dico :
            dico[code_teamA] = dA
        
        else :
            dictA= dico[code_teamA]
            for zone in dA.keys() :
                dictA[zone] = (dictA[zone][0]+dA[zone][0], dictA[zone][1]+dA[zone][1])
            dico[code_teamA] = dictA

        if code_teamB not in dico :
            dico[code_teamB] = dB
       
        else :
            dictB= dico[code_teamB]
            for zone in dB.keys() :
                dictB[zone] = (dictB[zone][0]+dB[zone][0], dictB[zone][1]+dB[zone][1])
            dico[code_teamB] = dictB
        
    return dico

Dans cette fonction on additione sur tous les matchs le nombre de tirs pris et réussis par zone dans un dictionnaire dont les clés sont les codes des équipes.

In [5]:
def get_fg_percentage(season) : 
    dico = get_fg_all_season(season)
    dico_percentage = {}

    for team in dico.keys() :
        dico_percentage[team] = {}
        
        for zone in dico[team].keys() :
            if dico[team][zone][1] != 0 :
                dico_percentage[team][zone] = dico[team][zone][0]/dico[team][zone][1]
    
    return dico_percentage

On calcule juste le pourcentage de réussite par zone correspondant aux nombre de tirs récolté auparavant.

In [6]:
def reduce_fg (season, gamecode, team) :
    dA, dB, code_teamA, code_teamB = get_fg(season, gamecode)
    dico = get_fg_percentage(season)
    dA_percentage, dB_percentage = {}, {}

    if team == code_teamA :
        for zone in dB.keys() :
            dB_percentage[zone] = dB[zone][0]/dB[zone][1] - dico[code_teamB][zone] if dB[zone][1] != 0 else 0
        return dB_percentage
    
    elif team == code_teamB :
        for zone in dA.keys() :
            dA_percentage[zone] = dA[zone][0]/dA[zone][1] - dico[code_teamA][zone] if dA[zone][1] != 0 else 0
        return dA_percentage

On s'intéresse ici à une équipe en particulier et à un de ses matchs. On regarde par zone de combien de pourcents le pourcentage de réussite au tir de l'équipe adverse a baissé ou augmenté lors de ce match comparé aux moyennes de saison.

In [7]:
def get_reduced_fg_all_season(season, team) : 
    gamecodes = get_camecodes(season)
    dico = {}

    for gamecode in gamecodes.keys() :
        if team in gamecodes[gamecode].values() : 
            d_percentage = reduce_fg(season, gamecode, team)
            for zone in d_percentage.keys() :
                if zone in dico:
                    dico[zone].append(d_percentage[zone])
                else:
                    dico[zone] = [d_percentage[zone]]

        else : 
            continue
    
    for zone in dico.keys() :
        dico[zone] = sum(dico[zone]) / len(dico[zone])
    
    return dico

On reprend la fonction précédente ici pour regarder une équipe particulière et faire la moyenne sur tous ses matchs de la saisons de l'écart de pourcentage par zone.

In [ ]:
def plot_defensive_heatmap(season, team): 
    dico_reduced = get_reduced_fg_all_season(season, team)
    zones_coords = {
        "C": {"x": 0, "y": 100},    # Sous le panier (Restricted Area)
        "D": {"x": -150, "y": 250}, # Raquette basse gauche
        "F": {"x": 150, "y": 250},  # Raquette basse droite
        "E": {"x": -150, "y": 450}, # Raquette haute gauche (High Post)
        "J": {"x": 150, "y": 450},  # Raquette haute droite
        "G": {"x": -350, "y": 100}, # Corner 3PT gauche
        "H": {"x": 350, "y": 100},  # Corner 3PT droit
        "I": {"x": -250, "y": 600}, # Aile 3PT gauche (Wing)
        "K": {"x": 250, "y": 600},  # Aile 3PT droite
        "L": {"x": 0, "y": 700}     # Axe central 3PT (Top of the key)
    }

    data = []
    for zone, reduction in dico_reduced.items():
        if zone in zones_coords:
            data.append({
                "zone": zone,
                "Reduction" : reduction,
                "x": zones_coords[zone]["x"],
                "y": zones_coords[zone]["y"],  
                "Texte_Affichage": f"{reduction*100:.1f}%"
            })

    df = pd.DataFrame(data)

    if df.empty:
        st.warning(f"Pas de données de tir trouvées pour {team}.")
        return None

    fig = px.scatter(
        df, 
        x="X", y="Y", 
        color="Reduction",
        text="Texte_Affichage",
        color_continuous_scale=px.colors.diverging.RdYlGn[::-1],
        range_color=[-0.15, 0.15], # Échelle fixe de -15% à +15%
        title=f"Heatmap Défensive : {team} (Stop Rate)"
    )

    fig.update_traces(
        marker=dict(size=45, line=dict(width=2, color='black')),
        textfont_color='black',
        textfont_size=13,
        textfont_family="Arial Black" # Pour que le texte ressorte bien
    )

    fig.add_layout_image(
        dict(
            source="terrain_euroleague.png", 
            xref="x", yref="y",
            x=-400,     # Centre l'image horizontalement
            y=850,      # Aligne le haut de l'image
            sizex=800,  # Largeur totale
            sizey=850,  # Hauteur totale
            sizing="stretch",
            opacity=0.6,
            layer="below"
        )
    )

    fig.update_xaxes(visible=False, range=[-400, 400])
    fig.update_yaxes(visible=False, range=[0, 850])

    fig.update_layout(
        plot_bgcolor="rgba(0,0,0,0)", 
        paper_bgcolor="rgba(0,0,0,0)",
        coloraxis_colorbar=dict(
            title="Variation d'Adresse",
            tickformat=".0%" # Affiche des % sur la barre de couleur
        )
    )

    return fig


Cette fonction fait un dataframe à partir des écarts de pourcentage calculés par zone puis le transforme en graphe. Pour cela on récupère un diagramme du demi-terrain d'Euroleague, on y fixe les coordonnées et on délimite chaque zone par les coordonnées correspondantes. Enfin à partir du dataframe on donne une couleur à chaque écart de pourcentage donc à chaque zone pour établir si la défense est bonne ou pas dans cette zone. 

In [9]:
def get_teams_code(season):
    gamecodes = get_camecodes(season)
    teams = []
    i = 0

    try :
        r = requests.get(f"{url}/shootingchart", params = {**credentials, "gamecode" : gamecode, "seasoncode" : f"E{season}"})
        r.raise_for_status()
        data = r.json()

        teamA = data["mainData"]["CodeTeamA"]
        teamB = data["mainData"]["CodeTeamB"]
        
        if not teamA in teams :
            teams.append(teamA)
            i+=1
        if not teamB in teams :
            teams.append(teamB)
            i+=1
        if i >=20 :
            return teams
    
    except Exception as e:
        print(f"Erreur lors de la récupération des données de tir : {e}")
        return teams

Cette dernière fonction sert juste à récupérer la liste des équipes de la saison sélectionnée, celle-ci changeant chaque année, afin de pouvoir permettre à l'utilisateur d'en sélectionner une pour afficher sa heatmap défensive.